### GraphRAG_with_LLM_and_neo4j

In [17]:
import os
from neo4j import GraphDatabase
import openai
from dotenv import load_dotenv


In [18]:
load_dotenv()

True

In [19]:
### Step 1:  Ser OpenAI API key for GPT bases LLM
openai.api_key = os.getenv("OPENAI_API_KEY")

In [20]:
### Step 2 : Connect to Neo4j Sandbox Cloud
uri = os.getenv("bolt_url")
username = os.getenv("username")
password = os.getenv("password")
driver = GraphDatabase.driver(uri, auth=(username, password))

In [21]:
### Step3: Initialize the Neo4j driver
driver = GraphDatabase.driver(uri, auth=(username, password))

### Step 4: Functions to Convert User Query to Neo4j Cypher Query, and Generate Response

In [26]:
### Convert the user query to Cypher query using GPT-4
def convert_to_cyper(user_query):
    prompt=f"Convert the following user query into a Cypher query to retrieve relevant movie data from Neo4j graph:\nUser Query: {user_query}\nCypher Query:"
    
    response = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[{'role':'user', 'content':prompt}],
        max_tokens=100,
        temperature=0.7
    )
    
    cypher_query = response.choices[0].message['content'].strip()
    # Extract the actual Cypher query by splitting on the first colon and taking the second part
    cypher_query = cypher_query.split(":", 1)[1].strip() # Extract only the query part
    
    # Check the query start with valid cypher clause and MATCH if needed
    
    if not cypher_query.startswith(('MATCH','MERGE','OPTIONAL MATCH')):
        cypher_query = "MATCH (a:" + cypher_query # Prepend 'MATCH (a:' to the query
    
    
    return cypher_query

In [27]:
# Query Neo4j using the generateed Cypher query

def query_with_cypher(cypher_query):
    with driver.session() as session:
        result = session.run(cypher_query)
        context = "\n".join([f"{record}" for record in result])
        
        return context

In [30]:
# query with context and generated response

def query_with_context_and_generate_response(user_query):
    # Convert user query to cypher
    cypher_query = convert_to_cyper(user_query)
    
    # Fetch relevant context using the cypher query
    context = query_with_cypher(cypher_query)
    
    # Create the prompt for gpt-4
    prompt = f"Context:\n{context}\n\nUser Query: {user_query}\nAnswer:"
    
    # Generate response form OpenAI GPT
    response = openai.ChatCompletion.create(
        model='gpt-4',
        messages=[{'role':'user', 'content':prompt}]
    )
    
    return response.choices[0].message['content']

### Run the demo with sample query

In [31]:
# Example User query
user_query = "Who acted in The Matrix?"

# Get response from GPT-4 with context from Neo4j

response = query_with_context_and_generate_response(user_query)
print("Response:", response)

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `ACTED_IN` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=19, offset=18>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 18, 'line': 1, 'column': 19}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (a:Actor)-[:ACTED_IN]->(m:Movie {title: 'The Matrix'}) RETURN a.name"
Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `Actor` does not exist in database `neo4j`. Verify that the spelling is correct.', positi

Response: The main actors in "The Matrix" include Keanu Reeves, Laurence Fishburne, Carrie-Anne Moss, and Hugo Weaving.
